# Music Success Analysis - Part 3: Platform Comparisons

This notebook focuses on comparing performance across different streaming platforms and analyzing time series patterns.

## Loading Libraries and Data

First, let's import the necessary libraries and load our cleaned dataset.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

# Set plotting style and ignore warnings
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

# Display settings for better visualization
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Load the cleaned dataset from pickle file
try:
    df = pd.read_pickle('cleaned_music_data.pkl')
    print("Loaded cleaned data from pickle file.")
except FileNotFoundError:
    print("Cleaned data file not found. Please run the '1_Data_Loading_Cleaning.ipynb' notebook first.")
    # If pickle file not found, load from CSV as fallback
    file_path = r"C:\Users\Adilf\Downloads\Most Streamed Spotify Songs 2024.csv (1)\Most Streamed Spotify Songs 2024.csv"
    df = pd.read_csv(file_path, encoding='latin1')
    print("Loaded original data from CSV file as fallback.")

# Display the first few rows of the dataset
df.head()

## Platform Comparisons

Let's compare performance across different streaming platforms.

In [ ]:
# Scatterplot comparing Spotify and YouTube
plt.figure(figsize=(12, 8))
sns.regplot(x='Spotify Streams', y='YouTube Views', data=df, 
           scatter_kws={'alpha':0.5, 's':50}, line_kws={'color':'red'})

plt.title('Spotify Streams vs YouTube Views', fontsize=16)
plt.xlabel('Spotify Streams', fontsize=12)
plt.ylabel('YouTube Views', fontsize=12)
plt.grid(linestyle='--', alpha=0.7)
plt.ticklabel_format(style='plain', axis='both')
plt.tight_layout()
plt.show()

# Calculate correlation coefficient
correlation = df['Spotify Streams'].corr(df['YouTube Views'])
print(f"Correlation between Spotify Streams and YouTube Views: {correlation:.4f}")

In [ ]:
# Scatterplot comparing Spotify and TikTok
plt.figure(figsize=(12, 8))
sns.regplot(x='Spotify Streams', y='TikTok Views', data=df, 
           scatter_kws={'alpha':0.5, 's':50, 'color':'purple'}, line_kws={'color':'darkviolet'})

plt.title('Spotify Streams vs TikTok Views', fontsize=16)
plt.xlabel('Spotify Streams', fontsize=12)
plt.ylabel('TikTok Views', fontsize=12)
plt.grid(linestyle='--', alpha=0.7)
plt.ticklabel_format(style='plain', axis='both')
plt.tight_layout()
plt.show()

# Calculate correlation coefficient
correlation = df['Spotify Streams'].corr(df['TikTok Views'])
print(f"Correlation between Spotify Streams and TikTok Views: {correlation:.4f}")

In [ ]:
# Scatterplot comparing YouTube and TikTok
plt.figure(figsize=(12, 8))
sns.regplot(x='YouTube Views', y='TikTok Views', data=df, 
           scatter_kws={'alpha':0.5, 's':50, 'color':'green'}, line_kws={'color':'darkgreen'})

plt.title('YouTube Views vs TikTok Views', fontsize=16)
plt.xlabel('YouTube Views', fontsize=12)
plt.ylabel('TikTok Views', fontsize=12)
plt.grid(linestyle='--', alpha=0.7)
plt.ticklabel_format(style='plain', axis='both')
plt.tight_layout()
plt.show()

# Calculate correlation coefficient
correlation = df['YouTube Views'].corr(df['TikTok Views'])
print(f"Correlation between YouTube Views and TikTok Views: {correlation:.4f}")

In [ ]:
# Cross-platform comparison for top tracks
# Get top 10 tracks by Spotify streams
top_tracks = df.nlargest(10, 'Spotify Streams')

# Normalize the values for better visualization
platforms = ['Spotify Streams', 'YouTube Views', 'TikTok Views']
normalized_data = pd.DataFrame()

for platform in platforms:
    max_val = df[platform].max()
    normalized_data[platform] = top_tracks[platform] / max_val

normalized_data['Track'] = top_tracks['Track']
normalized_data = normalized_data.set_index('Track')

# Create a heatmap of normalized values
plt.figure(figsize=(14, 8))
sns.heatmap(normalized_data, annot=True, cmap='YlGnBu', fmt='.2f', linewidths=0.5)
plt.title('Cross-Platform Performance of Top 10 Tracks (Normalized)', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Create a bar chart comparing platform performance for top 5 tracks
top_5_tracks = df.nlargest(5, 'Spotify Streams')

# Melt the dataframe for easier plotting
melted_df = pd.melt(top_5_tracks, 
                    id_vars=['Track'], 
                    value_vars=['Spotify Streams', 'YouTube Views', 'TikTok Views'],
                    var_name='Platform', value_name='Count')

# Create a grouped bar chart
plt.figure(figsize=(14, 8))
sns.barplot(x='Track', y='Count', hue='Platform', data=melted_df, palette='viridis')
plt.title('Platform Comparison for Top 5 Tracks', fontsize=16)
plt.xlabel('Track', fontsize=12)
plt.ylabel('Count (log scale)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yscale('log')  # Use log scale for better visualization
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='Platform')
plt.tight_layout()
plt.show()

## Time Series Analysis

Let's analyze temporal trends in music popularity.

In [ ]:
# Time series visualization
# Group by release month and calculate average streams
monthly_data = df.groupby('Release Month')['Spotify Streams'].agg(['count', 'mean'])
monthly_data.columns = ['Number of Releases', 'Average Streams']
monthly_data = monthly_data.reset_index()

# Create a figure with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Plot number of releases by month
sns.barplot(x='Release Month', y='Number of Releases', data=monthly_data, ax=ax1, color='skyblue')
ax1.set_title('Number of Releases by Month', fontsize=16)
ax1.set_ylabel('Number of Releases', fontsize=12)
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Plot average streams by month
sns.barplot(x='Release Month', y='Average Streams', data=monthly_data, ax=ax2, color='lightgreen')
ax2.set_title('Average Spotify Streams by Release Month', fontsize=16)
ax2.set_xlabel('Month', fontsize=12)
ax2.set_ylabel('Average Streams', fontsize=12)
ax2.grid(axis='y', linestyle='--', alpha=0.7)
ax2.ticklabel_format(style='plain', axis='y')

# Set month names as x-tick labels
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
ax2.set_xticklabels(month_names)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze release year trends
# Group by release year and calculate average streams
yearly_data = df.groupby('Release Year')['Spotify Streams'].agg(['count', 'mean'])
yearly_data.columns = ['Number of Releases', 'Average Streams']
yearly_data = yearly_data.reset_index()

# Filter out years with very few releases (likely outliers or data errors)
yearly_data = yearly_data[yearly_data['Number of Releases'] > 5]

# Create a figure with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Plot number of releases by year
sns.barplot(x='Release Year', y='Number of Releases', data=yearly_data, ax=ax1, color='skyblue')
ax1.set_title('Number of Releases by Year', fontsize=16)
ax1.set_ylabel('Number of Releases', fontsize=12)
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Plot average streams by year
sns.barplot(x='Release Year', y='Average Streams', data=yearly_data, ax=ax2, color='lightgreen')
ax2.set_title('Average Spotify Streams by Release Year', fontsize=16)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Average Streams', fontsize=12)
ax2.grid(axis='y', linestyle='--', alpha=0.7)
ax2.ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Create a heatmap of release month vs. platform performance
# Group by release month and calculate average performance for each platform
platforms = ['Spotify Streams', 'YouTube Views', 'TikTok Views']
month_platform_data = df.groupby('Release Month')[platforms].mean().reset_index()

# Normalize the values for better visualization
for platform in platforms:
    max_val = month_platform_data[platform].max()
    month_platform_data[f'{platform}_Normalized'] = month_platform_data[platform] / max_val

# Create a pivot table for the heatmap
normalized_platforms = [f'{platform}_Normalized' for platform in platforms]
heatmap_data = month_platform_data.pivot_table(index='Release Month', values=normalized_platforms)
heatmap_data.columns = [col.replace('_Normalized', '') for col in heatmap_data.columns]

# Create the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(heatmap_data, annot=True, cmap='YlGnBu', fmt='.2f', linewidths=0.5)
plt.title('Platform Performance by Release Month (Normalized)', fontsize=16)
plt.ylabel('Release Month', fontsize=12)
plt.yticks(ticks=np.arange(0.5, 12.5), labels=month_names)
plt.tight_layout()
plt.show()